# Communication Reward Penalty Ablation Evaluation

Evaluates policies trained with different `reward_comms_penalty_weight` values to characterize
the idleness/communication trade-off induced by the comms penalty. Companion to
`ptrain_patrolling_ablation_comms.py`, which sweeps the penalty weight while holding all other
training settings fixed (`merge_minimum_uq`, comms+pred+UQ stack).

Penalty weights swept (matches the training script):
- **0.0** — no penalty, comms are free
- **0.1**
- **0.2**
- **0.4**
- **0.5**
- **1.0**
- **10.0** — heavy penalty; expected to suppress comms aggressively

Sweeps observation radius with many eval episodes per (model, radius) for noise control.
Each episode resets the graph and spawn config, so episodes are independent draws.

## Configuration

In [1]:
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_ablation_comms_" + datetime_str + ".zarr"

# Pin output file once eval has been run, so re-running the notebook just loads results.
# output_file = "./results/patrolling_ablation_comms_test19.zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
from multiprocessing import Process
from args import *

MAX_PROCESSES = 200
GPUS = [0, 1, 2, 3, 4, 5, 6, 7]
GPU_AUTO_SELECT = True

def get_gpu_least_used():
    gpu_memory = os.popen("nvidia-smi --query-gpu=memory.used --format=csv").read()
    gpu_memory = gpu_memory.split("\n")[1:-1]
    gpu_memory = [int(x.split()[0]) for x in gpu_memory]
    gpu_memory = [gpu_memory[i] if i in GPUS else float('inf') for i in range(len(gpu_memory))]
    least_used_gpu = gpu_memory.index(min(gpu_memory))
    return least_used_gpu

# Default eval args. Bumped eval_episodes well above the original 3 — since each episode
# resets the graph + spawn config, episodes are independent and a single run with many
# episodes gives the same statistical power as a multi-seed sweep.
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(0)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(10)
default_args["episode_length"] = wrap_arg_val(150)
default_args["seed"] = wrap_arg_val(2)

default_args["observation_mask"] = wrap_arg_val(True)
default_args["graph_random"] = wrap_arg_val(True)

default_args["num_agents"] = wrap_arg_val(8)

# Fix the obs-radius randomization range so the swept observation_radius is what's actually used.
default_args["observation_radius_random_min"] = wrap_arg_val(0.0)
default_args["observation_radius_random_max"] = wrap_arg_val(0.0)

# Match training-time autoregression depth for the UQ-aware comms models.
default_args["diffusion_autoregression_steps"] = wrap_arg_val(7)

## Model Paths

Fill in the wandb run directories for each trained model. Each entry maps the comms penalty
weight to the `files/` directory of the corresponding wandb run from
`ptrain_patrolling_ablation_comms.py`.

Leave any entry as `None` to skip that model. Keys are floats (the penalty weight) so the
sweep stays sortable; the legend label is generated from the key.

In [3]:
# Map penalty weight -> wandb run files/ directory.
# Experiment names from ptrain_patrolling_ablation_comms.py:
#   patrolling_comms_pred_uq_commsPenalty{weight}
MODELS_BY_PENALTY = {
    0.0:  "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty0.0/wandb/run-20260525_001218-k9tygxal/files",
    0.1:  "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty0.1/wandb/run-20260525_001218-g0fxqcuw/files",
    0.2:  "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty0.2/wandb/run-20260525_001218-9ilc5zxp/files",
    0.4:  "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty0.4/wandb/run-20260525_001218-34v0x972/files",
    0.5:  "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty0.5/wandb/run-20260525_001218-znb3b20z/files",
    1.0:  "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty1.0/wandb/run-20260525_001218-w6xxadg2/files",
    10.0: "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty10.0/wandb/run-20260525_001218-fr5n5ses/files",
}

def penalty_label(w):
    # Stable, human-readable label that also sorts naturally as a series name.
    return f"Comms Penalty = {w:g}"

MODELS = {
    penalty_label(w): path
    for w, path in sorted(MODELS_BY_PENALTY.items())
    if path is not None
}
print(f"{len(MODELS)} model(s) configured:")
for name in MODELS:
    print(f"  - {name}")

7 model(s) configured:
  - Comms Penalty = 0
  - Comms Penalty = 0.1
  - Comms Penalty = 0.2
  - Comms Penalty = 0.4
  - Comms Penalty = 0.5
  - Comms Penalty = 1
  - Comms Penalty = 10


## Evaluation

In [4]:
import numpy as np

obs_radii = np.arange(5.0, 51.0, 5.0).tolist()

if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")
else:
    process_args = []

    def add_process(args):
        process_args.append(args)

    def start_process(args):
        p = Process(target=main, args=(args,))
        p.start()
        return p

    for series_name, model_dir in MODELS.items():
        for obs_radius in obs_radii:
            args = load_args(model_dir, default_args)
            args["eval_series"] = series_name
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            add_process(args)

    print(f"Queued {len(process_args)} eval runs across {len(MODELS)} model(s), "
          f"{len(obs_radii)} obs radii.")

    running_processes = []
    time_start = time.time()
    failed_processes = []

    while len(process_args) > 0 or len(running_processes) > 0:
        while len(running_processes) < MAX_PROCESSES and len(process_args) > 0:
            args = process_args.pop(0)
            if GPU_AUTO_SELECT:
                least_used_gpu = get_gpu_least_used()
                args["cuda_idx"] = wrap_arg_val(least_used_gpu)
                args["cuda_idx_predictor"] = wrap_arg_val(least_used_gpu)
            p = start_process(convert_args(args))
            running_processes.append(p)

        for p in running_processes:
            if not p.is_alive():
                p.join()
                print(f"Process {p.pid} finished with exit code {p.exitcode}. "
                      f"Running: {len(running_processes)-1}, queued: {len(process_args)}, "
                      f"failed: {len(failed_processes)}")
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]

    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")

    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")

    print("All processes finished successfully!")

Queued 70 eval runs across 7 model(s), 10 obs radii.
Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors_with_comm_uq_threshold', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--comm_uq_threshold_temperature', '0.1', '--communication_model', 'none', '--communi

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors_with_comm_uq_threshold', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--comm_uq_threshold_temperature', '0.1'

...

QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 1, available_action_dim: inf
  1: Categorical, action_dim: 1, available_action_dim: 10
R_Critic: Use CNN: False, Use MLP: True
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
R_Critic: Use CNN: False, Use MLP: True
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty10.0/wandb/run-20260525_001218-fr5n5ses/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty10.0/wandb/run-20260525_001218-fr5n5ses/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty10.0/wandb/run-20260525_001218-fr5n5ses/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling_comms_pred_uq_commsPenalty10.0/wandb/run-20260525_001218-fr5n5ses/files
Initialized QmasAlgorithm with 3 pre

## Analysis

In [5]:
import zarr
output = zarr.open(output_file, mode='r')
print(output.tree())
print(f"Output file: {os.path.abspath(output_file)}")
print(f"To pin results, set in first cell: output_file = '{os.path.abspath(output_file)}'")

/
 ├── Comms Penalty = 0
 │   ├── obs_radius_10.0
 │   │   ├── agent_count (10, 150) float32
 │   │   ├── avg_idleness (10, 150) float32
 │   │   ├── communication
 │   │   │   └── requests_made (10, 150) float32
 │   │   ├── deltaSteps (10, 150, 8, 1) float32
 │   │   ├── node_visits
 │   │   │   ├── node_0 (10, 150) float32
 │   │   │   ├── node_1 (10, 150) float32
 │   │   │   ├── node_10 (10, 150) float32
 │   │   │   ├── node_11 (10, 150) float32
 │   │   │   ├── node_12 (10, 150) float32
 │   │   │   ├── node_13 (10, 150) float32
 │   │   │   ├── node_14 (10, 150) float32
 │   │   │   ├── node_15 (10, 150) float32
 │   │   │   ├── node_16 (10, 150) float32
 │   │   │   ├── node_17 (10, 150) float32
 │   │   │   ├── node_18 (10, 150) float32
 │   │   │   ├── node_19 (10, 150) float32
 │   │   │   ├── node_2 (10, 150) float32
 │   │   │   ├── node_20 (10, 150) float32
 │   │   │   ├── node_21 (10, 150) float32
 │   │   │   ├── node_22 (10, 150) float32
 │   │   │   ├── node_23 (10,

In [6]:
import numpy as np

def episode_requests(data):
    """Total communication requests per episode (1-D array of length n_episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    return r.reshape(r.shape[0], -1).sum(axis=1)

def episode_idleness(data):
    """Time-averaged avg. idleness per episode (1-D array of length n_episodes)."""
    arr = np.array(data['avg_idleness'][...])
    # Reduce all non-episode axes to get one scalar per episode.
    if arr.ndim > 1:
        arr = arr.reshape(arr.shape[0], -1).mean(axis=1)
    return arr

# For each (series, obs_radius), pull per-episode arrays and compute mean + SEM.
summary = {}
for series in output:
    radii_data = {}
    for exp in output[series]:
        exp_data = output[series][exp]
        obs_r = float(exp_data.attrs.get('observation_radius', 0.0))
        radii_data[obs_r] = {
            'idleness': episode_idleness(exp_data),
            'requests': episode_requests(exp_data),
        }
    radii = sorted(radii_data.keys())
    summary[series] = {
        'obs_radius':     radii,
        'idleness_mean':  [float(radii_data[r]['idleness'].mean()) for r in radii],
        'idleness_sem':   [float(radii_data[r]['idleness'].std(ddof=1) / np.sqrt(len(radii_data[r]['idleness']))) for r in radii],
        'requests_mean':  [float(radii_data[r]['requests'].mean()) for r in radii],
        'requests_sem':   [float(radii_data[r]['requests'].std(ddof=1) / np.sqrt(len(radii_data[r]['requests']))) for r in radii],
        'n_episodes':     [int(len(radii_data[r]['idleness'])) for r in radii],
    }
    print(f"{series}: {len(radii)} obs radii, n_episodes per point: {summary[series]['n_episodes']}")

Comms Penalty = 0: 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Comms Penalty = 0.1: 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Comms Penalty = 0.2: 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Comms Penalty = 0.4: 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Comms Penalty = 0.5: 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Comms Penalty = 1: 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Comms Penalty = 10: 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]


In [7]:
import re
import plotly.express as px

def _series_penalty(name):
    m = re.search(r"([0-9]*\.?[0-9]+)", name)
    return float(m.group(1)) if m else float('inf')

# Sort series by penalty weight so plots/legend read low -> high.
series_order = sorted(summary.keys(), key=_series_penalty)

# Use a sequential colorscale (low penalty -> high penalty) instead of qualitative colors,
# since the series form an ordered sweep.
n = len(series_order)
if n > 1:
    sampled = px.colors.sample_colorscale('Viridis', [i / (n - 1) for i in range(n)])
else:
    sampled = px.colors.sample_colorscale('Viridis', [0.5])
series_colors = dict(zip(series_order, sampled))

### Idleness vs. Observation Radius

In [8]:
import plotly.graph_objects as go

def _rgb_to_rgba(rgb_str, alpha=0.2):
    # plotly sample_colorscale returns 'rgb(r, g, b)' strings.
    nums = re.findall(r"[0-9.]+", rgb_str)
    r, g, b = nums[0], nums[1], nums[2]
    return f'rgba({r},{g},{b},{alpha})'

fig = go.Figure()
for series in series_order:
    color = series_colors[series]
    s = summary[series]
    xs = np.array(s['obs_radius'])
    ys = np.array(s['idleness_mean'])
    es = np.array(s['idleness_sem'])

    fig.add_trace(go.Scatter(
        x=np.concatenate([xs, xs[::-1]]),
        y=np.concatenate([ys + es, (ys - es)[::-1]]),
        fill='toself',
        fillcolor=_rgb_to_rgba(color, alpha=0.15),
        line=dict(color='rgba(255,255,255,0)'),
        legendgroup=series, showlegend=False, hoverinfo='skip',
    ))
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines+markers',
        name=series, line=dict(color=color), marker=dict(color=color, size=8),
        legendgroup=series,
        hovertemplate=f"{series}<br>obs_radius=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>",
    ))

fig.update_layout(
    title="Time-Averaged Avg. Idleness vs. Observation Radius (shaded: ±1 SEM across episodes)",
    xaxis_title="Observation Radius",
    yaxis_title="Time-Averaged Avg. Idleness (s)",
    height=600, width=900, hovermode='closest', autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()

### Communication Rate vs. Observation Radius

In [9]:
fig = go.Figure()
for series in series_order:
    color = series_colors[series]
    s = summary[series]
    xs = np.array(s['obs_radius'])
    ys = np.array(s['requests_mean'])
    es = np.array(s['requests_sem'])

    fig.add_trace(go.Scatter(
        x=np.concatenate([xs, xs[::-1]]),
        y=np.concatenate([ys + es, (ys - es)[::-1]]),
        fill='toself',
        fillcolor=_rgb_to_rgba(color, alpha=0.15),
        line=dict(color='rgba(255,255,255,0)'),
        legendgroup=series, showlegend=False, hoverinfo='skip',
    ))
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines+markers',
        name=series, line=dict(color=color), marker=dict(color=color, size=8),
        legendgroup=series,
        hovertemplate=f"{series}<br>obs_radius=%{{x:.1f}}<br>requests=%{{y:.1f}}<extra></extra>",
    ))

fig.update_layout(
    title="Total Communication Requests vs. Observation Radius (shaded: ±1 SEM across episodes)",
    xaxis_title="Observation Radius",
    yaxis_title="Total Communication Requests (per episode)",
    height=600, width=900, hovermode='closest', autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()

### Pareto: Idleness vs. Communication Rate

Each penalty weight traces a curve through (comm rate, idleness) space as observation radius
varies. The headline result of the ablation: a higher penalty weight should shift the curve
toward fewer comms — at the cost of higher idleness — and the union of curves traces an
empirical Pareto frontier across the penalty sweep.

In [10]:
fig = go.Figure()
markers = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'star']
for series in series_order:
    color = series_colors[series]
    s = summary[series]
    xs = np.array(s['requests_mean'])
    ys = np.array(s['idleness_mean'])
    radii = s['obs_radius']

    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='markers+text',
        name=series, line=dict(color=color), marker=dict(color=color, size=8, symbol=markers[series_order.index(series) % len(markers)]),
        legendgroup=series,
        # text=[f"{r:.0f}" for r in radii],
        textposition='top center',
        textfont=dict(color=color, size=9),
        hovertemplate=[
            f"{series}<br>obs_radius={r:.1f}<br>requests=%{{x:.2f}}<br>idleness=%{{y:.4f}}<extra></extra>"
            for r in radii
        ],
    ))

fig.update_layout(
    title="Idleness vs. Communication",
    xaxis_title="Total Communication Requests (per episode)",
    yaxis_title="Time-Averaged Avg. Idleness (s)",
    height=600, width=900, hovermode='closest', autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()

### Penalty Sweep: Idleness and Comm Rate vs. Penalty Weight

Ablation-specific view: at each observation radius, plot idleness and comm rate as a function
of the penalty weight. This makes the monotonicity of the trade-off (or lack thereof) directly
visible — the x-axis is the swept hyperparameter, not the eval setting.

In [11]:
from plotly.subplots import make_subplots

# Pivot summary by observation radius -> per-radius series across penalty weights.
all_radii = sorted({r for s in summary.values() for r in s['obs_radius']})
penalties = [_series_penalty(s) for s in series_order]

radius_colors = dict(zip(
    all_radii,
    px.colors.sample_colorscale('Plasma', [i / max(len(all_radii) - 1, 1) for i in range(len(all_radii))]),
))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Idleness vs. Comms Penalty Weight", "Comm Requests vs. Comms Penalty Weight"),
    shared_xaxes=False,
)

for r in all_radii:
    color = radius_colors[r]
    ws, idl, req = [], [], []
    for series, w in zip(series_order, penalties):
        s = summary[series]
        if r in s['obs_radius']:
            i = s['obs_radius'].index(r)
            ws.append(w)
            idl.append(s['idleness_mean'][i])
            req.append(s['requests_mean'][i])
    fig.add_trace(go.Scatter(
        x=ws, y=idl, mode='lines+markers',
        name=f"obs_radius={r:g}", legendgroup=f"r{r}",
        line=dict(color=color), marker=dict(color=color, size=7),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=ws, y=req, mode='lines+markers',
        name=f"obs_radius={r:g}", legendgroup=f"r{r}", showlegend=False,
        line=dict(color=color), marker=dict(color=color, size=7),
    ), row=1, col=2)

fig.update_xaxes(title_text="Comms Penalty Weight", type='log', row=1, col=1)
fig.update_xaxes(title_text="Comms Penalty Weight", type='log', row=1, col=2)
fig.update_yaxes(title_text="Time-Averaged Avg. Idleness (s)", row=1, col=1)
fig.update_yaxes(title_text="Total Comm Requests (per episode)", row=1, col=2)

fig.update_layout(
    height=500, width=1100, hovermode='closest',
    margin=dict(l=60, r=20, t=60, b=80),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()

### Summary Table

In [12]:
import pandas as pd
rows = []
for series in series_order:
    s = summary[series]
    for i, r in enumerate(s['obs_radius']):
        rows.append({
            'series': series,
            'penalty_weight': _series_penalty(series),
            'obs_radius': r,
            'idleness_mean': s['idleness_mean'][i],
            'idleness_sem':  s['idleness_sem'][i],
            'requests_mean': s['requests_mean'][i],
            'requests_sem':  s['requests_sem'][i],
            'n_episodes':    s['n_episodes'][i],
        })
df = pd.DataFrame(rows)
df

ModuleNotFoundError: No module named 'pandas'